In [1]:
from fulltext_search import SearchEngine

In [3]:
process_id = "e167a74e-ca82-458f-8890-844acf7a99e1"
internal_pass_header = "JPgLCxSqeF9nT3S3Bt1Pp206oF4MKffr9CV6O3Y2bBDAkv3URDZKs64x85PevJY7fJe6MSSWcBi"
engine = SearchEngine.from_paging_api(
    url="http://localhost:8080/api/v1/process-instances/{pid}/tasks".format(pid=process_id),
    headers={"x-internal-pass": internal_pass_header},
)
print("indexed documents:", engine.document_count)

rewrite: AC378 "Germany Entity" "Bundesbank Files" task completion instructions email required OR notification OR confirmation OR closeout procedure
ambiguous Send monthly deliverables to financial reporting Document(id='694047f7-5ced-40e2-8ae5-5f8c49d60deb', content="Code: P72-6f8c01255121-T45\nTitle: Send monthly deliverables to financial reporting\nState: NOT_READY\nDescription: Inputs: quarterly financials, monthly financials, and year end financials.\n  Process: send to Reg team\n  Outputs: sent mails with finanical reports\nPreconditions: {'id': '1cc48c51-7a35-4fc7-b980-a8a93759f2f7', 'text': 'PIA start the task once books are reviewed and closed in BD6', 'satisfied': False}; {'id': 'ff3c9a60-47e9-49a4-b4e3-27b3bde88e97', 'text': 'PIA start the task once books are reviewed and closed in BD6', 'satisfied': False}; {'id': 'd096648a-a9cf-4448-9b25-d99e1b5e16fc', 'text': 'PIA start the task once books are reviewed and closed in BD6', 'satisfied': False}; {'id': '11d03d81-114e-447f-97

In [ ]:
## Top-k Corrective RAG (`ask`)

Returns the best `top_k` records only (relevant, ambiguous, then irrelevant if needed to fill the list).

In [ ]:
query = "Should I send an email to complete the Germany Entity Bundesbank Files task (AC378)?"

result = engine.ask(query, top_k=5)
if result.rewritten_query:
    print("rewrite:", result.rewritten_query)
for hit, record in zip(result.results, result.records):
    print(hit.metadata.get("relevance"), "-", record.metadata.get("title"))

## Pageable brute-force search (`search_all`)

Grades **every** indexed document (multithreaded), keeps only records graded `relevant`, ranks them by lexical score, and persists the full result set to a temp-file session. The first page is returned immediately; page further with `get_search_page` (no re-grading).

In [ ]:
page = engine.search_all(query, page=0, size=5)

print("session:", page.session_id)
if page.rewritten_query:
    print("rewrite:", page.rewritten_query)
print(f"total relevant: {page.total_elements} across {page.total_pages} page(s)\n")

for hit in page.content:
    title = hit.document.metadata.get("title") if hit.document else hit.document_id
    print(f"[{hit.metadata.get('lexical_score'):.3f}] {title}")
    print("   reason:", hit.metadata.get("rationale"))

In [ ]:
# Walk every page of the same session without re-running the LLM grading.
session_id = page.session_id
size = 5
current = engine.get_search_page(session_id, page=0, size=size)

rank = 0
while True:
    for hit in current.content:
        rank += 1
        title = hit.document.metadata.get("title") if hit.document else hit.document_id
        print(f"{rank:>3}. {title}")
    if current.last:
        break
    current = engine.get_search_page(session_id, page=current.page + 1, size=size)